In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image as ShowImage, display

%matplotlib inline
plt.rcParams.update({'font.size': 14})

# the toy models are tiny: the CPU is fast enough and gives the same numbers everywhere
device = 'cpu'

SAVE_FIGURES = False  # set to True to write the figures used on the slides to ./figures
def savefig(name):
    if SAVE_FIGURES:
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')

def save_anim(anim, name, fps):
    # write a matplotlib animation as a GIF and show it
    path = f'figures/{name}.gif' if SAVE_FIGURES else f'{name}.gif'
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    anim.save(path, writer=PillowWriter(fps=fps))
    plt.close()
    display(ShowImage(filename=path))

In [ ]:
%%sh
# Download the data - you need to do this only once
wget --no-verbose --output-document=image_dog.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_dog.jpg

# Lecture 16 - Generative Models: Flow Matching

We build a flow-matching model from scratch for a 2D toy distribution (a smiley), where we can plot everything:

1. **ODEs and Euler's method**: how a velocity field moves points.
2. **Flow matching**: learn the velocity field that moves Gaussian noise to the data.
3. **How many steps?** What the network learns is an *average*, and few steps give blurry samples.
4. **Classifier-free guidance**: a conditional model, pushed towards its condition.
5. **The same path on an image**: what $x_t$ looks like for a photo.

Everything runs on a CPU in a few minutes. Run this notebook on Google Colab (see the [readme](https://github.com/chrirupp/cv_course)).

## 1. Ordinary differential equations

An ODE $\frac{dx}{dt} = f(x, t)$ says how fast and in which direction $x$ moves. Our example is $\frac{dx}{dt} = -x$, with the solution $x(t) = x(0)\, e^{-t}$.

In [ ]:
t = np.linspace(0, 5, 200)
plt.figure(figsize=(9, 6))
for x0 in [-3, -2, -1, 0.5, 1, 2, 3]:
    plt.plot(t, x0 * np.exp(-t), linewidth=2.5, label=f'$x(0) = {x0}$')
plt.axhline(0, color='k', linewidth=0.5)
plt.xlabel('$t$')
plt.ylabel('$x(t)$')
plt.title(r'Solutions of $dx/dt = -x$: $x(t) = x(0)\,e^{-t}$')
plt.legend(fontsize=11, loc='upper right')
plt.grid(alpha=0.3)
savefig('gen_ode')
plt.show()

**Euler's method** follows the velocity in small steps: $x_{t+h} \approx x_t + h\, f(x_t, t)$.

In [ ]:
def euler(f, x0, t_end, h):
    ts, xs = [0.0], [x0]
    while ts[-1] < t_end - 1e-9:
        xs.append(xs[-1] + h * f(xs[-1], ts[-1]))
        ts.append(ts[-1] + h)
    return np.array(ts), np.array(xs)

f = lambda x, t: -x
x0, h = 3.0, 0.5
t_exact = np.linspace(0, 5, 200)
ts, xs = euler(f, x0, 5, h)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(t_exact, x0 * np.exp(-t_exact), 'k-', linewidth=2.5, alpha=0.5, label=r'exact: $x(0)\,e^{-t}$')
line, = ax.plot([], [], 'o-', color='tab:red', linewidth=2.5, markersize=8, label=f'Euler, $h = {h}$')
arrow = ax.annotate('', xy=(0, 0), xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='tab:green', linewidth=3))
info = ax.text(0.97, 0.55, '', transform=ax.transAxes, ha='right', fontsize=15)
ax.set_xlim(0, 5); ax.set_ylim(-0.3, 3.3)
ax.set_xlabel('$t$'); ax.set_ylabel('$x$')
ax.legend(loc='upper right'); ax.grid(alpha=0.3)
ax.set_title(r"Euler's method for $dx/dt = -x$")

def frame(k):
    k = min(k, len(ts) - 1)
    line.set_data(ts[:k + 1], xs[:k + 1])
    if k < len(ts) - 1:     # the next step: follow the slope at the current point
        arrow.set_position((ts[k], xs[k]))
        arrow.xy = (ts[k + 1], xs[k + 1])
    info.set_text(f'step {k}:  $x$ = {xs[k]:.3f}, exact {x0 * np.exp(-ts[k]):.3f}')
    return line, arrow, info

save_anim(FuncAnimation(fig, frame, frames=len(ts) + 3, interval=700), 'gen_euler', fps=1.5)

In [ ]:
# smaller steps: closer to the exact solution, but more evaluations of f
plt.figure(figsize=(9, 6))
plt.plot(t_exact, x0 * np.exp(-t_exact), 'k-', linewidth=3, alpha=0.5, label='exact')
for h_, c in zip([1.0, 0.5, 0.2, 0.05], ['tab:red', 'tab:orange', 'tab:blue', 'tab:green']):
    ts_, xs_ = euler(f, x0, 5, h_)
    plt.plot(ts_, xs_, 'o-', color=c, markersize=4, label=f'$h$ = {h_} ({len(ts_) - 1} steps)')
plt.xlabel('$t$'); plt.ylabel('$x$'); plt.legend(); plt.grid(alpha=0.3)
plt.title("Euler's method: effect of the step size")
savefig('gen_euler_steps')
plt.show()

## 2. Flow matching in 2D

The data distribution is a smiley: a ring (80% of the points), two eyes and a mouth. Images live in $\mathbb{R}^{3 \times H \times W}$; here we learn a distribution in $\mathbb{R}^2$. The source is Gaussian noise. Each data point also gets a label (ring, eyes, mouth), which we only use in Section 4.

In [ ]:
def create_dataset(n, radius=2.5, rng=None):
    rng = rng or np.random.default_rng(0)
    n_ring, n_eyes = n * 8 // 10, n // 20
    a = rng.uniform(0, 2 * np.pi, n_ring)
    ring = radius * np.stack([np.cos(a), np.sin(a)], 1) + rng.normal(0, 0.1, (n_ring, 2))
    centres = np.where(rng.random(n_eyes)[:, None] < 0.5, [[-1, 1]], [[1, 1]])
    eyes = centres + rng.normal(0, 0.1, (n_eyes, 2))
    n_mouth = n - n_ring - n_eyes
    a = rng.uniform(np.pi, 2 * np.pi, n_mouth)
    mouth = 0.5 * radius * np.stack([np.cos(a), np.sin(a)], 1) + rng.normal(0, 0.05, (n_mouth, 2))
    labels = np.concatenate([np.zeros(n_ring), np.ones(n_eyes), 2 * np.ones(n_mouth)]).astype(int)
    return np.concatenate([ring, eyes, mouth]).astype(np.float32), labels

data, labels = create_dataset(20000)
PART = ['ring', 'eyes', 'mouth']
SIGMA = 2.0                     # the noise: x0 ~ N(0, SIGMA^2 I)
rng = np.random.default_rng(1)
noise = rng.normal(0, SIGMA, (5000, 2))

def setup(ax, title=None):
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    if title: ax.set_title(title)

for name, pts, c, title in [('gen_noise', noise, 'tab:gray', 'noise: $x_0 \\sim \\mathcal{N}(0, \\sigma^2 I)$'),
                            ('gen_data', data, 'tab:blue', 'data: $x_1 \\sim p_{data}$')]:
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.3, c=c)
    setup(ax, title)
    savefig(name)
    plt.show()

**Straight paths.** For one noise sample $x_0$ and one data sample $x_1$, the path $x_t = (1 - t)\, x_0 + t\, x_1$ is a straight line with constant velocity $x_1 - x_0$. Training pairs random noise with random data, so the lines cross: many different velocities pass through the same point.

In [ ]:
rng = np.random.default_rng(3)
inside = np.flatnonzero(np.abs(noise).max(1) < 4)          # noise samples that fit in the plot
i0, i1 = rng.choice(inside, 12, replace=False), rng.choice(len(data), 12, replace=False)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(noise[:, 0], noise[:, 1], s=2, alpha=0.12, c='tab:gray')
ax.scatter(data[:, 0], data[:, 1], s=2, alpha=0.12, c='tab:blue')
for a, b in zip(noise[i0], data[i1]):
    ax.plot([a[0], b[0]], [a[1], b[1]], '-', color='tab:red', linewidth=1.5, alpha=0.8)
    ax.plot(*a, 'o', color='k', markersize=5)
    ax.plot(*b, 'o', color='tab:blue', markersize=6, markeredgecolor='k')
    ax.plot(*(0.5 * a + 0.5 * b), 'o', color='tab:red', markersize=5)
setup(ax, 'pairs $(x_0, x_1)$ and $x_{0.5}$ (red)')
savefig('gen_pairs')
plt.show()

**The network** $f(x, t)$ is a small MLP. As in the lecture's neural fields, its inputs (here $x$, $y$ and $t$) are encoded with sines and cosines at several frequencies. A label input (one of ring, eyes, mouth, or "none") is used only by the conditional model of Section 4.

In [ ]:
class VelocityField(torch.nn.Module):
    def __init__(self, hidden=128, n_freq=5, n_labels=0):
        super().__init__()
        self.n_freq, self.n_labels = n_freq, n_labels
        d_in = 3 * 3 * n_freq + (n_labels + 1 if n_labels else 0)
        self.net = torch.nn.Sequential(
            torch.nn.Linear(d_in, hidden), torch.nn.ReLU(),
            torch.nn.Linear(hidden, hidden), torch.nn.ReLU(),
            torch.nn.Linear(hidden, hidden), torch.nn.ReLU(),
            torch.nn.Linear(hidden, 2))

    def encode(self, v):
        # [batch, 3] -> [batch, 3 * 3 * n_freq]: v, sin(2^k v), cos(2^k v)
        v = v[..., None] * 2.0 ** torch.arange(self.n_freq, dtype=v.dtype)
        return torch.cat([v, torch.sin(v), torch.cos(v)], -1).flatten(1)

    def forward(self, x, t, label=None):
        inp = [self.encode(torch.cat([x, t[:, None]], -1))]
        if self.n_labels:   # one-hot label; index n_labels means "no label"
            inp.append(torch.nn.functional.one_hot(label, self.n_labels + 1).float())
        return self.net(torch.cat(inp, -1))

def train(model, steps, batch=512, p_drop=None, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    X, Y = torch.tensor(data), torch.tensor(labels)
    for it in (tq := tqdm(range(steps))):
        idx = torch.randint(0, len(X), (batch,))
        x1 = X[idx]                                  # data samples
        x0 = torch.randn(batch, 2) * SIGMA           # noise samples
        t = torch.rand(batch)                        # random times in [0, 1]
        xt = (1 - t[:, None]) * x0 + t[:, None] * x1 # a point on the straight path
        label = None
        if p_drop is not None:                       # classifier-free guidance: sometimes drop the label
            label = torch.where(torch.rand(batch) < p_drop, model.n_labels, Y[idx])
        loss = ((model(xt, t, label) - (x1 - x0)) ** 2).sum(-1).mean()
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if it % 100 == 0:
            tq.set_postfix(loss=f'{loss.item():.3f}')
    return model.eval()

model = train(VelocityField(), 6000)

**Sampling** integrates $\frac{dx}{dt} = f(x, t)$ from $t = 0$ to $1$ with Euler's method.

In [ ]:
@torch.no_grad()
def sample(f, x0, n_steps):
    """Euler steps from t = 0 to 1; f(x, t) is the velocity. Returns the whole trajectory."""
    x, h, traj = x0.clone(), 1.0 / n_steps, [x0.numpy()]
    for i in range(n_steps):
        x = x + h * f(x, torch.full((len(x),), i * h))
        traj.append(x.numpy())
    return np.stack(traj)

torch.manual_seed(1)
x0 = torch.randn(400, 2) * SIGMA
traj = sample(model, x0, 100)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(data[:, 0], data[:, 1], s=2, alpha=0.1, c='tab:blue')
lines = [ax.plot([], [], '-', color='tab:red', alpha=0.25, linewidth=1)[0] for _ in range(len(x0))]
dots = ax.scatter([], [], s=12, c='tab:red')
setup(ax)

def frame(k):
    k = min(k, len(traj) - 1)
    for i, l in enumerate(lines):
        l.set_data(traj[:k + 1, i, 0], traj[:k + 1, i, 1])
    dots.set_offsets(traj[k])
    ax.set_title(f'samples, t = {k / (len(traj) - 1):.2f}')
    return lines + [dots]

save_anim(FuncAnimation(fig, frame, frames=range(0, len(traj) + 20, 2), interval=60), 'gen_trajectories', fps=15)

**The learned velocity field** at five times (colour: speed). Early on (small $t$) the noise says little about where a point will end, and all arrows point roughly to the centre of the data. Later the field becomes specific: it pulls points onto the ring, the eyes and the mouth.

In [ ]:
g = np.linspace(-4.5, 4.5, 23)
GX, GY = np.meshgrid(g, g)
grid = torch.tensor(np.stack([GX.ravel(), GY.ravel()], 1), dtype=torch.float32)

def plot_field(ax, f, t_val, title=None):
    with torch.no_grad():
        v = f(grid, torch.full((len(grid),), t_val)).numpy()
    ax.scatter(data[:, 0], data[:, 1], s=1, alpha=0.06, c='tab:blue')
    speed = np.linalg.norm(v, axis=1)
    v = v * (np.minimum(speed, 4) / np.maximum(speed, 1e-6))[:, None]   # arrow length clipped, colour = speed
    ax.quiver(GX, GY, v[:, 0].reshape(GX.shape), v[:, 1].reshape(GX.shape), np.minimum(speed, 8), cmap='plasma',
              angles='xy', scale_units='xy', scale=9, width=0.006, clim=(0, 8))
    setup(ax, title or f't = {t_val:.2f}')

# t = 1 itself is never used by Euler's method (the last step starts at 1 - h), and the field is ill-defined there
for t_val in [0.0, 0.25, 0.5, 0.75, 0.95]:
    fig, ax = plt.subplots(figsize=(5, 5))
    plot_field(ax, model, t_val)
    savefig(f'gen_field_t{t_val:.2f}')
    plt.show()

In [ ]:
# the field changing over time (left) next to the samples it moves (right)
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
ts_anim = np.linspace(0, 1, 41)
traj_a = sample(model, x0, 40)      # traj_a[k] is the sample at time ts_anim[k]

def frame(k):
    k = min(k, len(ts_anim) - 1)
    for ax in axes: ax.clear()
    plot_field(axes[0], model, ts_anim[k], f'velocity field, t = {ts_anim[k]:.2f}')
    axes[1].scatter(data[:, 0], data[:, 1], s=1, alpha=0.08, c='tab:blue')
    axes[1].scatter(traj_a[k, :, 0], traj_a[k, :, 1], s=10, c='tab:red')
    setup(axes[1], f'samples, t = {ts_anim[k]:.2f}')
    return []

save_anim(FuncAnimation(fig, frame, frames=len(ts_anim) + 8, interval=120), 'gen_field', fps=8)

## 3. How many steps?

Training pairs every noise sample with every data sample, so the network learns the *average* velocity of all straight paths through a point: $f(x, t) = \mathbb{E}[x_1 - x_0 \mid x_t = x]$. Each conditional path is straight, but this average field is not: the samples' paths curve (see the animation above). Euler's method needs several steps to follow curved paths. We start from the *same* noise and use 1, 2, 5, 20 and 100 steps.

In [ ]:
from scipy.spatial import cKDTree
tree = cKDTree(data)
torch.manual_seed(2)
x0 = torch.randn(1000, 2) * SIGMA
STEPS = [1, 2, 5, 20, 100]
for n in STEPS:
    xs = sample(model, x0, n)[-1]
    d, _ = tree.query(xs)
    print(f'{n:3d} steps: median distance to the nearest data point {np.median(d):.2f}, '
          f'mean distance from the centre {np.linalg.norm(xs, axis=1).mean():.2f}')
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(data[:, 0], data[:, 1], s=1, alpha=0.08, c='tab:blue')
    ax.scatter(xs[:, 0], xs[:, 1], s=6, c='tab:red', alpha=0.6)
    setup(ax, f'{n} step' + ('s' if n > 1 else ''))
    savefig(f'gen_steps_{n}')
    plt.show()
print('ring radius 2.5; data points are on average', f'{np.linalg.norm(data, axis=1).mean():.2f}', 'from the centre')

## 4. Classifier-free guidance

A **conditional** model gets a label $c$ (ring, eyes or mouth) as an extra input: $f(x, t, c)$. Classifier-free guidance trains the *same* network without the label 10% of the time ($c = \varnothing$). At sampling time it extrapolates from the unconditional towards the conditional velocity:

$$\tilde f = f(x, t, \varnothing) + w\, \big(f(x, t, c) - f(x, t, \varnothing)\big)$$

$w = 0$ is the unconditional model, $w = 1$ the conditional one, and $w > 1$ pushes the samples further towards what distinguishes the label.

In [ ]:
cond = train(VelocityField(n_labels=3), 10000, p_drop=0.1, seed=1)

def guided(c, w):
    def f(x, t):
        null = torch.full((len(x),), 3)
        lab = torch.full((len(x),), c)
        v_u = cond(x, t, null)
        return v_u + w * (cond(x, t, lab) - v_u)
    return f

EYES = 1
centres = np.array([[-1, 1], [1, 1]])
torch.manual_seed(3)
x0 = torch.randn(600, 2) * SIGMA
for w in [0, 1, 4]:
    xs = sample(guided(EYES, w), x0, 100)[-1]
    d = np.min(np.linalg.norm(xs[:, None] - centres[None], axis=-1), axis=1)
    near = d < 0.5
    print(f'w = {w}: {100 * near.mean():.0f}% of the samples within 0.5 of an eye centre, '
          f'their RMS distance to it {np.sqrt(np.mean(d[near] ** 2)):.3f} (data: {0.1 * np.sqrt(2):.3f})')
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(data[:, 0], data[:, 1], s=1, alpha=0.08, c='tab:blue')
    ax.scatter(xs[:, 0], xs[:, 1], s=6, c='tab:red', alpha=0.6)
    setup(ax, {0: 'w = 0 (unconditional)', 1: 'w = 1 ("eyes")', 4: 'w = 4 ("eyes")'}[w])
    savefig(f'gen_cfg_w{w}')
    plt.show()

## 5. The same path on an image

For images, $x_1$ is a photo and $x_0$ Gaussian noise of the same size (here per pixel and channel, values in $[-1, 1]$). The straight path blends them: at $t = 0.5$ the dog is already visible through the noise. A generative model runs this backwards, from $t = 0$ (pure noise) to $t = 1$, one velocity step at a time.

In [ ]:
img = cv2.cvtColor(cv2.imread('image_dog.jpg'), cv2.COLOR_BGR2RGB)
img = cv2.resize(img[:, 150:150 + 853], (256, 256), interpolation=cv2.INTER_AREA)
x1 = img.astype(np.float32) / 127.5 - 1
x0 = np.random.default_rng(0).normal(0, 1, x1.shape).astype(np.float32)
for k, t_val in enumerate([0.0, 0.25, 0.5, 0.75, 1.0]):
    xt = (1 - t_val) * x0 + t_val * x1
    plt.figure(figsize=(3, 3))
    plt.imshow(np.clip((xt + 1) / 2, 0, 1))
    plt.axis('off')
    plt.title(f't = {t_val:.2f}')
    savefig(f'gen_path_{k}')
    plt.show()